---
# 3. Data Analysis & Visualisations

In this section, we will be conducting Data Analysis and Visualisations on the Data provided to better understand the Dataset provided to us. Some of the Analysis will provide insights on areas to address to ensure our model performs better and much more consistent.

---
## 3.1 Addressing Training Data Abnormalies

In this sub-section, we will be addressing the Data Abnormally of the Testing Data where there are presence of Carrots in the Beans Folder. This Abnormally was identified during a manual inspection of the data. Thus, re-directing the data is the best course of action to undertake in this case as without doing so will cause a contemination in Data between Carrot and Bean.

In [ ]:
# ======================== Setup ======================== #
filenames_to_remove = [
    '0001', '0002', '0003', '0004',
    '0017', '0018', '0019', '0020',
    '0033', '0049', '0050'
]
filenames_to_remove = [f + '.jpg' for f in filenames_to_remove]

source_dir = os.path.join(train_dir, 'Bean')
target_dir = os.path.join(train_dir, 'Radish and Carrot')
os.makedirs(target_dir, exist_ok=True)

# ======================== Visualise & Move ======================== #
moved_files = []

for fname in filenames_to_remove:
    src_path = os.path.join(source_dir, fname)
    tgt_path = os.path.join(target_dir, fname)

    if not os.path.exists(src_path):
        print(f"File not found: {fname}")
        continue

    # ----- Display Image ----- #
    img = cv2.imread(src_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(4, 4))
    plt.imshow(img_rgb)
    plt.title(f"Moving: {fname}")
    plt.axis('off')
    plt.show()

    # ----- Move File ----- #
    shutil.move(src_path, tgt_path)
    moved_files.append(fname)
    print(f"Moved: {fname} → Radish and Carrot")

# ======================== Summary ======================== #
print(f"\nSuccessfully moved {len(moved_files)} files.")

With reference to the output above, we have successfully addresed the abnormally.

---
## 3.2 Visualising Class Imbalance

In this sub-section, we will be visualising the quantity count for each of the vegetables class so as to identify any potential Class Imbalance. Class Imbalance could cause skewed accuracy as the model is able to better learn from the vegetable class with higher quantity of data whereas vegetable classes with lower number of data will cause a lower accuracy. This can cause a high accuracy model as the model may be extremely capable of identifying a specific class while failing to identify other classes. The various actions that can be taken to address Class Imbalance are listed below:

**Methods To Address Class Imbalance**
- Establishing Class Weights
- Intense Augmentation of Minority Classes
- Oversampling / Undersampling

With the various Methods to Address Class Imbalance, we shall proceed to state the appropriate action plan for various outcomes of the visualisation. The action plan for each respective results are listed below:

**Class Imbalance Identified**
- List down the observations from the Barplot.
- Proceed to address Class Imbalance using one of the listed method.
- Justify the chosen method's usage over other method.

**No Class Imbalance Identified**
- List down the observations from the Barplot.
- Proceed to next visualisation.

With the action plan for the potential results listed, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ======================== 1. Dataset Directory Map ======================== #
dataset_dirs = {
    'Training': train_dir,
    'Validation': val_dir,
    'Testing': test_dir
}

# ======================== 2. Class Count Aggregation ======================== #
records = []

for split_name, split_path in dataset_dirs.items():
    class_names = sorted(os.listdir(split_path))
    for cls in class_names:
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            records.append({
                'Dataset': split_name,
                'Vegetable': cls,
                'Image Count': count
            })

# ======================== 3. DataFrame Setup ======================== #
df_counts = pd.DataFrame(records)
ordered_veggies = sorted(df_counts['Vegetable'].unique())
df_counts['Vegetable'] = pd.Categorical(df_counts['Vegetable'], categories=ordered_veggies, ordered=True)

# ======================== 4. Polished Plotly Bar Plots ======================== #
for split in ['Training', 'Validation', 'Testing']:
    subset = df_counts[df_counts['Dataset'] == split]

    fig = px.bar(
        subset,
        x='Vegetable',
        y='Image Count',
        color='Vegetable',
        text='Image Count',
        title=f"{split} Set – Class Distribution",
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    fig.update_traces(
        textposition='outside',
        marker_line_width=0.5,
        marker_line_color='black'
    )

    fig.update_layout(
        width=1600,
        height=500,
        plot_bgcolor='rgba(248,248,255,1)',
        paper_bgcolor='white',
        title_font=dict(size=20, family='Arial Black'),
        font=dict(size=13),
        margin=dict(l=20, r=20, t=70, b=70),
        xaxis=dict(
            title="Vegetable Class",
            tickangle=45,
            titlefont=dict(size=14),
            tickfont=dict(size=12),
            gridcolor='lightgray'
        ),
        yaxis=dict(
            title="Number of Images",
            titlefont=dict(size=14),
            tickfont=dict(size=12),
            gridcolor='lightgray'
        ),
        showlegend=False
    )

    fig.show()

With reference to the output above, we are able to observe that there are **Class Imbalance Identified** within the Training Dataset. Only the Training Dataset will matter in this case as Training Data needs to be balanced and consistent as the other Datasets are already balanced. The observations from the Barplot are listed below:

**Observations**
- Number of Data Varies for each Class of Vegetables.
- Tomato has the highest data count at 955.
- Capsicum has the lowest data count at 351.

With the observations indicated and listed above, we will proceed to address the Class Imbalance. The addressment will only be done to the Training Data.

---
### 3.2.1 Addressing Class Imbalance Using Class Weights

With reference to the previous sub-section, we will be using the **Class Weights** Method to address the Class Imbalance within our Training Data. The usage of the Class Weights will be listed below:

**Applications & Implications**
- Tells the model to assign higher penalty to error on rare classes.
- Causes the model to learn from both Minority and Majority classes.

With the Application and Implications listed, we will proceed to list out the reasoning behind selecting the Class Weights method instead of other methods:

**Reason for Utilising Class Weights**
- No data duplication required.
- Works directly with Keras.
- No amplification of noisy minority samples.
- Commonly utilised in CNNs.

With the Reasoning listed, it is also of utmost importance to recognise that Class Weights do not have any direct impact of Data Quantity but rather only the Training Process. The formula that we will utilise for calculating the Class Weights is listed below:

$$
w_i = \frac{N}{C \cdot n_i}
$$

Where:
- $N$ = total number of samples  
- $C$ = number of classes  
- $n_i$ = number of samples in class $i$
- $w_i$ = Class Weight

With the formula indicated, we shall proceed to calculate the Class Weight in the Code Cell below.

In [ ]:
# ======================== Getting Class Names From Generator ======================== #
class_indices = train_generator_23.class_indices
class_names = list(class_indices.keys())

# ======================== Image Count per Class ======================== #
class_counts = [len(os.listdir(os.path.join(train_dir, cls))) for cls in class_names]
total_images = sum(class_counts)
num_classes = len(class_names)

# ======================== Computing Class Weights ======================== #
class_weight_dict = {
    i: total_images / (num_classes * count)
    for i, count in enumerate(class_counts)
}

# ======================== Creating of DataFrame ======================== #
df_class_weights = pd.DataFrame({
    'Class': class_names,
    'Image Count': class_counts,
    'Class Weight': [class_weight_dict[class_indices[cls]] for cls in class_names]
})

# ======================== Displaying of DataFrame ======================== #
df_class_weights.style.background_gradient(cmap="Blues")

With reference to the output above, we are able to confirm that the Class Weights of the various Classes have been successfully computed and are ready to be utilised for training.

---
## 3.3 Visualising Sample Images

In this sub-section, we will be visualising sample images from each of the Vegetable Class within the Dataset provided. Through this visualisation, we will also be able to confirm that the specifications stipulated by the instructions for the image pre-processing have also been achieved. This will also allow us to visualise the input for the models.

---
### 3.3.1 Visualising Sample Images (23px by 23px)

In the Code Cell below, we will be visualising the Training Images for the 23px by 23px. We will be able to see the quality and attributes of the Images that will be input into the CNN Model through this visualisation.

In [ ]:
# ======================== Defining Parameters ======================== #
sample_size = (23, 23)
num_classes = len(class_names)

# ======================== Calculating Optimal Rows & Cols ======================== #
cols = 6
rows = (num_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

for i, cls in enumerate(class_names):
    img_path = os.path.join(train_dir, cls, os.listdir(os.path.join(train_dir, cls))[0])
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, sample_size)
    axes[i].imshow(img_resized, cmap='gray')
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Subplots ======================== #
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# ======================== Title & Layout ======================== #
plt.suptitle(f"Sample Grayscale Images from Each Class ({sample_size[0]}×{sample_size[1]})",
             fontsize=16, weight='bold')
plt.show()

With refence to the output of the images above, we are able to observe that the images are extremely unclear and distorted due to the low resolution requirement. This could imply that the models that will be trained on the 23px by 23px dataset will most likely have a low accuracy due to the small amount of pixel coverage.

---
### 3.3.2 Visualising Sample Images (101px by 101px)

In the Code Cell below, we will be visualising the Training Images for the 101px by 101px. We will be able to see the quality and attributes of the Images that will be input into the CNN Model through this visualisation.

In [ ]:
# ======================== Defining Parameters ======================== #
sample_size = (101, 101)
num_classes = len(class_names)

# ======================== Calculating Optimal Rows & Cols ======================== #
cols = 6
rows = (num_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

for i, cls in enumerate(class_names):
    img_path = os.path.join(train_dir, cls, os.listdir(os.path.join(train_dir, cls))[0])
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, sample_size)
    axes[i].imshow(img_resized, cmap='gray')
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Subplots ======================== #
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# ======================== Title & Layout ======================== #
plt.suptitle(f"Sample Grayscale Images from Each Class ({sample_size[0]}×{sample_size[1]})",
             fontsize=16, weight='bold')
plt.show()

With refence to the output of the images above, we are able to observe that the images are generally clear due to the higher resolution requirement. This could imply that the models that will be trained on the 101px by 101px dataset will most likely have a higher accuracy due to the larger amount of pixel coverage as compared to the Image with the 23px by 23px resolution.

---
### 3.3.3 Visualising Augmented Sample Images (23px by 23px)

In the Code Cell below, we will be visualising the Augmented Training Images for the 23px by 23px. We will be able to see the quality and attributes of the Images that will be input into the CNN Model through this visualisation.

In [ ]:
# ======================== Identifying Class Names ======================== #
class_labels = {v: k for k, v in train_generator_23_aug.class_indices.items()}
collected = {}

# ======================== Looping Through All Classes ======================== #
while len(collected) < len(class_labels):
    x_batch, y_batch = next(train_generator_23_aug)
    for img, label_vec in zip(x_batch, y_batch):
        class_index = np.argmax(label_vec)
        class_name = class_labels[class_index]
        if class_name not in collected:
            collected[class_name] = img
        if len(collected) == len(class_labels):
            break

# ======================== Plotting of Images ======================== #
cols = 6
rows = (len(collected) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

for i, (class_name, img) in enumerate(sorted(collected.items())):
    axes[i].imshow(img.reshape(23, 23), cmap='gray')
    axes[i].set_title(class_name, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Axes ======================== #
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# ======================== Title & Layout ======================== #
plt.suptitle("Augmented Image Samples (23x23)", fontsize=16, weight='bold')
plt.show()

With refence to the output of the images above, we are able to observe that the images are extremely unclear and distorted due to the low resolution requirement. This could imply that the models that will be trained on the Augmented 23px by 23px dataset will most likely have a low accuracy due to the small amount of pixel coverage.

---
### 3.3.4 Visualising Augmented Sample Images (101px by 101px)

In the Code Cell below, we will be visualising the Augmented Training Images for the 101px by 101px. We will be able to see the quality and attributes of the Images that will be input into the CNN Model through this visualisation.

In [ ]:
# ======================== Identifying Class Names ======================== #
class_labels = {v: k for k, v in train_generator_101_aug.class_indices.items()}
collected = {}

# ======================== Looping Through All Classes ======================== #
while len(collected) < len(class_labels):
    x_batch, y_batch = next(train_generator_101_aug)
    for img, label_vec in zip(x_batch, y_batch):
        class_index = np.argmax(label_vec)
        class_name = class_labels[class_index]
        if class_name not in collected:
            collected[class_name] = img
        if len(collected) == len(class_labels):
            break

# ======================== Plotting of Images ======================== #
cols = 6
rows = (len(collected) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

for i, (class_name, img) in enumerate(sorted(collected.items())):
    axes[i].imshow(img.reshape(101, 101), cmap='gray')
    axes[i].set_title(class_name, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Axes ======================== #
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# ======================== Title & Layout ======================== #
plt.suptitle("Augmented Image Samples (One per Class, 101×101)", fontsize=16, weight='bold')
plt.show()

With refence to the output of the images above, we are able to observe that the images are generally clear due to the higher resolution requirement. This could imply that the models that will be trained on the 101px by 101px dataset will most likely have a higher accuracy due to the larger amount of pixel coverage as compared to the Image with the 23px by 23px resolution.

---
## 3.4 Visualising Mean Images

In this sub-section, we will be visualing the Mean Images for the Training Data provided. This will allow us to identify Class Distinctiveness, Assess our Data Quality, and understand Class Similarities. The Mean Images are gathered using a mathematical formula indicated below:

- $I_c^{(i)}(x, y)$ be the $i^{th}$ image of class $c$ at pixel position $(x, y)$  
- $n$ be the number of images in class $c$

Then the **mean image** for class $c$ is computed as:

$$
\text{MeanImage}_c(x, y) = \frac{1}{n} \sum_{i=1}^{n} I_c^{(i)}(x, y)
$$

With the Mathematical Formula indicated, we will proceed to obtain the Mean Image for each of the Vegetable Class.

---
### 3.4.1 Visualising Mean Images (23px by 23px)

We will be visualising the Mean Images for the 23px by 23px Image Data in the Code Cell below. Subsequently, we will attempt to interpret any potential Distinct Characteristics of each of the Vegetable Classes and at the same time attempt to indentify any Class Similarities so as to potentially identify any Vegetable Classes that may be mixed up by the model.

In [ ]:
# ======================== Defining Parameters ======================== #
sample_size = (23, 23)
num_classes = len(class_names)
cols = 6
rows = (num_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

# ======================== Plotting of Mean Images ======================== #
for i, cls in enumerate(class_names):
    imgs = []
    cls_folder = os.path.join(train_dir, cls)
    for img_file in os.listdir(cls_folder)[:30]:
        img = cv2.imread(os.path.join(cls_folder, img_file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, sample_size)
        imgs.append(img)
    mean_img = np.mean(imgs, axis=0)
    axes[i].imshow(mean_img, cmap='gray')
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Subplots ======================== #
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# ======================== Title & Layout ======================== #
plt.suptitle(f"Mean Image per Class ({sample_size[0]}×{sample_size[1]})",
             fontsize=16, weight='bold')
plt.show()

With reference to the output above, we are able to observe some Class Distinct Characteristics. However, due to the low image quality of 23px by 23px, most of the Visualisations appears to be noise. The observations are indicated below:

**Bean**
- Mildly Blurry and Noisy.
- Consist of a Singular Dark 'Dot' in the Middle.

**Bitter_Gourd**
- Extremely Noisy.
- No observations can be made.

**Brinjal**
- Mildly Blurry and Noisy
- Consist of Darker Spots in the Botton Right Corner.
- High Brightness in the Botton Left Region.

**Cabbage**
- Mildly Blurry and Noisy.
- High Central Brightness.

**Capsicum**
- Mildly Blurry and Noisy.
- Consist of a Dark Areas in Bottom Left Regions.

**Cauliflower and Broccoli**
- Mildly Blurry and Noisy.
- High Brightness in Central Region.
- May potentially Cause Mix-Up with Other Classes (e.g. Raddist and Carrot)

**Cucumber and Bottle_Gourd**
- Extremely Noisy.
- No observations can be made.

**Potato**
- Relatively Clear.
- Circle-shaped Dark Spot can be observed in the Central Region.

**Pumpkin**
- Extremely Noisy.
- No observations can be made.

**Raddish and Carrot**
- Mildly Blurry and Noisy.
- High Brightness in Central Region.
- May potentially Cause Mix-Up with Other Classes (e.g. Cauliflower and Broccoli)

**Tomato**
- Extremely Noisy.
- No observations can be made.

With the observations listed, we will proceed to visualise the Mean Image for the Image Data with the resolution of 101px by 101px.

---
### 3.4.2 Visualising Mean Images (101px by 101px)

We will be visualising the Mean Images for the 101px by 101px Image Data in the Code Cell below. Subsequently, we will attempt to interpret any potential Distinct Characteristics of each of the Vegetable Classes and at the same time attempt to indentify any Class Similarities so as to potentially identify any Vegetable Classes that may be mixed up by the model.

In [ ]:
# ======================== Defining Parameters ======================== #
sample_size = (101, 101)
num_classes = len(class_names)
cols = 6
rows = (num_classes + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

# ======================== Plotting of Mean Images ======================== #
for i, cls in enumerate(class_names):
    imgs = []
    cls_folder = os.path.join(train_dir, cls)
    for img_file in os.listdir(cls_folder)[:30]:
        img = cv2.imread(os.path.join(cls_folder, img_file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, sample_size)
        imgs.append(img)
    mean_img = np.mean(imgs, axis=0)
    axes[i].imshow(mean_img, cmap='gray')
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Subplots ======================== #
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# ======================== Title & Layout ======================== #
plt.suptitle(f"Mean Image per Class (Resized to {sample_size[0]}×{sample_size[1]})",
             fontsize=16, weight='bold')
plt.show()

With reference to the output above, we are able to observe some Class Distinct Characteristics. However, due to the low image quality of 23px by 23px, most of the Visualisations appears to be noise. The observations are indicated below:

**Bean**
- Mildly Blurry and Noisy.
- Consist of a Singular Dark 'Dot' in the Middle.

**Bitter_Gourd**
- Extremely Noisy.
- No observations can be made.

**Brinjal**
- Mildly Blurry and Noisy
- Consist of Darker Spots in the Botton Right Corner.
- High Brightness in the Botton Left Region.

**Cabbage**
- Mildly Blurry and Noisy.
- High Central Brightness.

**Capsicum**
- Mildly Blurry and Noisy.
- Consist of a Dark Areas in Bottom Left Regions.

**Cauliflower and Broccoli**
- Mildly Blurry and Noisy.
- High Brightness in Central Region.
- May potentially Cause Mix-Up with Other Classes (e.g. Raddist and Carrot)

**Cucumber and Bottle_Gourd**
- Mildly Blurry and Noisy.
- High Central Brightness in an ellipse shape.

**Potato**
- Relatively Clear.
- Circle-shaped Dark Spot can be observed in the Central Region.

**Pumpkin**
- Extremely Noisy.
- No observations can be made.

**Raddish and Carrot**
- Mildly Blurry and Noisy.
- High Brightness in Central Region.
- May potentially Cause Mix-Up with Other Classes (e.g. Cauliflower and Broccoli)

**Tomato**
- Extremely Noisy.
- No observations can be made.

With the observations listed, we are able to discover that for Mean Image Visualisations, higher resolution may or may not incur a change in observations.

---
## 3.5 Identifying Abnormalies in Validation Data

In this sub-section, we will be visualising the Validation Data to identify any potential Abnormalies within the Validation Data. These Abnormalies may include, incorrect naming, inconsistent labeling, incorrect assignment. Subsequently should there be any abnormalies identified, we will proceed with the addressment. The action plan are listed below for the various scenarios:

**Abnormalies Identified**
- Identify the Abnormally.
- Address the Abnormally using the most appropriate action.

**No Abnormalies Identified**
- Proceed to Identify Potential Abnormalies within the Testing Data.

With the action plan listed, we will proceed to visualise the Validation Data in Coloured Settings so as to allow easier Identification.

In [ ]:
# ======================== Getting Class Names from Folders ======================== #
class_names = sorted([
    name for name in os.listdir(val_dir)
    if os.path.isdir(os.path.join(val_dir, name)) and len(os.listdir(os.path.join(val_dir, name))) > 0
])
num_classes = len(class_names)

# ======================== Calculating Grid Size ======================== #
cols = 6
rows = (num_classes + cols - 1) // cols

# ======================== Plotting ======================== #
fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

for i, cls in enumerate(class_names):
    cls_folder = os.path.join(val_dir, cls)
    img_files = os.listdir(cls_folder)
    img_path = os.path.join(cls_folder, img_files[0])
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, sample_size)

    # ----- Converting BGR To RGB Before Plotting ----- #
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

    axes[i].imshow(img_rgb)
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Subplots ======================== #
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    f"Validation Data Images from Each Class ({sample_size[0]}×{sample_size[1]})",
    fontsize=16, weight='bold'
)
plt.tight_layout()
plt.show()

With reference to the output of the, we are able to identify that there are **Abnormalies Identified** as there are Wrong Naming Conventions as compared to the Training Data. The Abnormalies changes are listed below:

- Cauliflower with Broccoli (Validation Data) ⇒ Cauliflower and Broccoli (Training Data)

- Cucumber with Bottle_Gourd (Validation Data) ⇒ Cucumber and Bottle_Gourd (Training Data)

- Radish with Carrot (Validation Data) ⇒ Radish and Carrot (Training Data)

With the abnormalies identified, we will proceed to address the Naming Convention in the next Code Cell.

---
### 3.5.1 Addressing Incorrect Naming Convention

As previously identified, there are inconsistent naming conventions throughout the Validation Dataset. Therefore, we will correct the Data Naming and ensure all Data Labels are consistent with the Data Label of the Training Data.

In [ ]:
# ======================== Folder Renaming Mappings ======================== #
folder_renames = {
    "Cucumber with Bottle_Gourd": "Cucumber and Bottle_Gourd",
    "Cauliflower with Broccoli": "Cauliflower and Broccoli",
    "Radish with Carrot": "Radish and Carrot"
}

# ======================== Performing Safe Renaming ======================== #
for old_name, new_name in folder_renames.items():
    old_path = os.path.join(val_dir, old_name)
    new_path = os.path.join(val_dir, new_name)

    if os.path.exists(old_path) and not os.path.exists(new_path):
        os.rename(old_path, new_path)
        print(f"Renamed: '{old_name}' → '{new_name}'")
    elif os.path.exists(new_path):
        print(f"Already renamed: '{new_name}' exists.")
    else:
        print(f"Skipped: '{old_name}' does not exist.")

With reference to the output above, we are able to confirm that the Labels for the Validation Dataset have been successfully corrected in accordance with the Naming Convention of the Training Data.

---
## 3.6 Identifying Abnormalies in Testing Data

In this sub-section, we will be visualising the Testing Data to identify any potential Abnormalies within the Testing Data. These Abnormalies may include, incorrect naming, inconsistent labeling, incorrect assignment. Subsequently should there be any abnormalies identified, we will proceed with the addressment. The action plan are listed below for the various scenarios:

**Abnormalies Identified**
- Identify the Abnormally.
- Address the Abnormally using the most appropriate action.

**No Abnormalies Identified**
- Proceed to Identify Potential Abnormalies within the Testing Data.

With the action plan listed, we will proceed to visualise the Testing Data in Coloured Settings so as to allow easier Identification.

In [ ]:
# ======================== Plotting with correct colours ======================== #
fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
axes = axes.flatten()

for i, cls in enumerate(class_names):
    img_path = os.path.join(test_dir, cls, os.listdir(os.path.join(test_dir, cls))[0])
    img = cv2.imread(img_path)
    img = cv2.resize(img, sample_size)

    # ----- Converting BGR To RGB Before Plotting ----- #
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')

# ======================== Removing Unused Subplots ======================== #
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(f"Validation Data Images ({sample_size[0]}×{sample_size[1]})",
             fontsize=16, weight='bold')
plt.tight_layout()
plt.show()

With reference to the output of the, we are able to identify that there are **Abnormalies Identified** as there are wrong Naming Conventions and Incorrect Labelling as compared to the Training Data. The Abnormalies Changes are listed below:

- Bottle_Gourd and Cucumber (Testing Data) ⇒ Cucumber and Bottle_Gourd (Training Data)

- Broccoli and Cauliflower (Testing Data) ⇒ Cauliflower and Broccoli (Training Data)

- Capsicum (Apparently) (Testing Data) ⇒ Capsicum (Training Data)

- Carrot and Radish (Testing Data) ⇒ Radish and Carrot (Training Data)

- Pumpkin (purportedly) ⇒ Pumpkin

- Tomato (ostensibly) ⇒ Tomato

- Pumpkin ⇔ Tomato

With the abnormalies identified, we will proceed to address the Naming Convention in the next Code Cell.

---
### 3.6.1 Addressing Incorrect Naming Conventions & Incorrect Labellings

As previously identified, there are inconsistent Naming Conventions and Incorrect Labellings throughout the Testing Dataset. Therefore, we will correct the Data Naming and ensure all Data Labels are consistent with the Data Label of the Training Data.

In [ ]:
# ======================== Folder Renaming Mapping ======================== #
test_renames = {
    "Pumpkin (purportedly)": "Tomato",
    "Tomato (ostensibly)": "Pumpkin",
    "Capsicum (apparently)": "Capsicum",
    "Bottle_Gourd and Cucumber": "Cucumber and Bottle_Gourd",
    "Broccoli and Cauliflower": "Cauliflower and Broccoli",
    "Carrot and Radish": "Radish and Carrot"
}

# ======================== Performing Safe Renaming ======================== #
for old_name, new_name in test_renames.items():
    old_path = os.path.join(test_dir, old_name)
    new_path = os.path.join(test_dir, new_name)

    if os.path.exists(old_path) and not os.path.exists(new_path):
        os.rename(old_path, new_path)
        print(f"Renamed: '{old_name}' → '{new_name}'")
    elif os.path.exists(new_path):
        print(f"Already renamed: '{new_name}' exists.")
    else:
        print(f"Skipped: '{old_name}' does not exist.")

With reference to the output above, we are able to confirm that the Labels for the Testing Dataset have been successfully corrected in accordance with the Naming Convention and Labelling of the Training Data.